# Bond Math

This notebook covers the fundamentals of fixed-income pricing using `bond.core`.

**Topics**
- Price, yield, and the price-yield relationship
- Macaulay and modified duration
- Convexity and the price-yield approximation
- DV01 and sensitivity analysis
- Accrued interest and clean vs dirty price

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yourusername/bond/blob/main/notebooks/01_bond_math.ipynb)

In [ ]:
# Uncomment to install from PyPI / GitHub
# !pip install bond-math
# or from local clone:
import sys; sys.path.insert(0, '.')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from bond import (
    price, ytm, duration, modified_duration,
    convexity, dv01, accrued_interest, clean_price,
    yield_to_call, price_change_approx,
)

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
print('bond.core loaded ✓')

## 1. Bond Pricing

A bond's dirty (full) price is the present value of all future cash flows discounted at the yield to maturity:

$$P = \sum_{i=1}^{n} \frac{C}{(1 + y/f)^i} + \frac{F}{(1 + y/f)^n}$$

where $C = F \cdot c / f$ is the periodic coupon, $f$ is the payment frequency, and $y$ is the annual YTM.

In [ ]:
# A 5% semi-annual 10-year bond at various yields
face, coupon, maturity, freq = 1000, 0.05, 10, 2

scenarios = {
    'Discount (YTM 6%)': price(face, coupon, maturity, 0.06, freq),
    'Par (YTM = coupon 5%)': price(face, coupon, maturity, 0.05, freq),
    'Premium (YTM 4%)': price(face, coupon, maturity, 0.04, freq),
}
for name, p in scenarios.items():
    print(f'{name:30s}  ${p:>10.4f}')

In [ ]:
# Price-yield relationship
yields = np.linspace(0.01, 0.12, 200)
prices = [price(1000, 0.05, 10, y, 2) for y in yields]

plt.figure()
plt.plot(yields * 100, prices, lw=2, color='#1565C0')
plt.axvline(5, ls='--', color='gray', lw=1, label='Coupon rate = 5%')
plt.axhline(1000, ls='--', color='gray', lw=1, label='Par = $1000')
plt.xlabel('Yield to Maturity (%)')
plt.ylabel('Price ($)')
plt.title('Price-Yield Relationship — 5% Semi-annual 10Y Bond')
plt.legend()
plt.tight_layout()
plt.show()

## 2. Yield to Maturity

YTM is the internal rate of return — the single discount rate that makes the PV of all cash flows equal to the market price. It is solved numerically.

In [ ]:
# Suppose a 5% semi-annual 10Y bond trades at $1,050
market_price = 1050.0
y = ytm(face=1000, coupon=0.05, maturity=10, price=market_price, frequency=2)
print(f'Market price: ${market_price}  →  YTM: {y:.4%}')

# Verify round-trip
p_check = price(1000, 0.05, 10, y, 2)
print(f'Round-trip check: price at YTM = ${p_check:.4f}')

## 3. Duration

**Macaulay Duration**: weighted average time to receive cash flows.

$$D_{mac} = \frac{\sum_i t_i \cdot PV(CF_i)}{P}$$

**Modified Duration**: measures price sensitivity to yield changes.

$$D_{mod} = \frac{D_{mac}}{1 + y/f}$$

$$\frac{\Delta P}{P} \approx -D_{mod} \cdot \Delta y$$

In [ ]:
maturities = [1, 2, 3, 5, 7, 10, 15, 20, 30]
mac_durs = [duration(1000, 0.05, m, 0.05, 2) for m in maturities]
mod_durs = [modified_duration(1000, 0.05, m, 0.05, 2) for m in maturities]

fig, ax = plt.subplots()
ax.plot(maturities, mac_durs, 'o-', lw=2, label='Macaulay Duration')
ax.plot(maturities, mod_durs, 's--', lw=2, label='Modified Duration')
ax.plot(maturities, maturities, ':', color='gray', lw=1, label='Zero-coupon bound')
ax.set_xlabel('Maturity (years)')
ax.set_ylabel('Duration (years)')
ax.set_title('Duration vs Maturity — 5% Semi-annual Bond at Par')
ax.legend()
plt.tight_layout()
plt.show()

print(f"\n10Y bond: Mac Duration = {duration(1000,0.05,10,0.05,2):.4f}y, "
      f"Mod Duration = {modified_duration(1000,0.05,10,0.05,2):.4f}y")

## 4. Convexity

Convexity is the second-order price sensitivity. It explains why the price-yield curve is *convex*: for large yield moves, the duration approximation understates the actual price change.

$$\frac{\Delta P}{P} \approx -D_{mod} \cdot \Delta y + \frac{1}{2} \cdot C \cdot (\Delta y)^2$$

In [ ]:
y0 = 0.05
p0 = price(1000, 0.05, 10, y0, 2)
mod = modified_duration(1000, 0.05, 10, y0, 2)
cx = convexity(1000, 0.05, 10, y0, 2)

print(f'Price at y=5%:         ${p0:.4f}')
print(f'Modified Duration:      {mod:.4f}')
print(f'Convexity:              {cx:.4f}')

dy_range = np.linspace(-0.03, 0.03, 200)
actual_prices = [price(1000, 0.05, 10, y0 + dy, 2) for dy in dy_range]
duration_approx = [p0 + price_change_approx(p0, mod, 0, dy) for dy in dy_range]
duration_convexity_approx = [p0 + price_change_approx(p0, mod, cx, dy) for dy in dy_range]

fig, ax = plt.subplots()
ax.plot(dy_range * 100, actual_prices, lw=2.5, label='Actual price', color='#1565C0')
ax.plot(dy_range * 100, duration_approx, '--', lw=1.5, label='Duration only', color='#E53935')
ax.plot(dy_range * 100, duration_convexity_approx, '--', lw=1.5, label='Duration + Convexity', color='#2E7D32')
ax.set_xlabel('Yield Change (bp × 100)')
ax.set_ylabel('Price ($)')
ax.set_title('Price Approximation: Duration vs Duration + Convexity')
ax.legend()
plt.tight_layout()
plt.show()

## 5. DV01 — Dollar Value of 1bp

DV01 (also called PVBP) is the dollar price change for a 1 basis point move in yield. It is the most common risk measure used by traders.

In [ ]:
maturities = [1, 2, 3, 5, 7, 10, 15, 20, 30]
dv01s = [dv01(1000, 0.05, m, 0.05, 2) for m in maturities]

fig, ax = plt.subplots()
ax.bar(maturities, dv01s, color='#1565C0', alpha=0.8, width=0.8)
ax.set_xlabel('Maturity (years)')
ax.set_ylabel('DV01 ($)')
ax.set_title('DV01 by Maturity — $1,000 face, 5% coupon at par')
for m, d in zip(maturities, dv01s):
    ax.text(m, d + 0.1, f'${d:.2f}', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.show()

print('\nDV01 table:')
print(f'{"Maturity":>10}  {"DV01 ($)":>10}')
for m, d in zip(maturities, dv01s):
    print(f'{m:>10}y  {d:>10.4f}')

## 6. Accrued Interest & Clean Price

When a bond trades between coupon dates, the buyer compensates the seller for the coupon that has accrued since the last payment date.

- **Dirty price** = Full price (what you actually pay)
- **Clean price** = Dirty price − Accrued Interest (what's quoted in the market)

In [ ]:
face, coupon, mat, y, freq = 1000, 0.05, 10, 0.05, 2
dirty = price(face, coupon, mat, y, freq)

fractions = np.linspace(0, 1, 100)
dirty_prices = [dirty] * 100  # dirty price stays constant (assuming yield constant)
ai_values = [accrued_interest(face, coupon, freq, t_since_last=t) for t in fractions]
clean_prices = [dirty - ai for ai in ai_values]

fig, ax = plt.subplots()
ax.plot(fractions, dirty_prices, lw=2, color='#1565C0', label='Dirty price')
ax.plot(fractions, clean_prices, lw=2, color='#2E7D32', label='Clean price')
ax.fill_between(fractions, clean_prices, dirty_prices, alpha=0.15, color='orange', label='Accrued interest')
ax.set_xlabel('Fraction of coupon period elapsed')
ax.set_ylabel('Price ($)')
ax.set_title('Clean vs Dirty Price Through a Coupon Period')
ax.legend()
plt.tight_layout()
plt.show()

t_mid = 0.5
print(f'Halfway through coupon period:')
print(f'  Dirty price:       ${dirty:.4f}')
print(f'  Accrued interest:  ${accrued_interest(face, coupon, freq, t_since_last=t_mid):.4f}')
print(f'  Clean price:       ${clean_price(face, coupon, mat, y, freq, t_since_last=t_mid):.4f}')